# Zeitliche Analyse: Kongruenz und Abstimmungsaktivität

Auswertung **über die Zeit**: Wie entwickeln sich Übereinstimmung, Empfehlungsaktivität und Rechtsformen von 1848 bis heute?

**Was passiert hier?**
- Kongruenz-Datensatz laden und historische Phasen kurz kontrollieren
- Balkenplots: Anzahl Abstimmungen und Anteil abgegebener Empfehlungen
- Linien- und Boxplots: BR, BV und Parteien vs. Volk über Jahrzehnt/Legislatur
- Rechtsformen im Zeitverlauf
- Interaktive Plotly-Grafik für den Blog

**Datengrundlage**
- `data/processed/df_with_positions.csv` (aus `2_berechnung.ipynb`, inkl. `phase`, `zustimmung_*`, `jahrzehnt`)

**Vorher ausführen**
- `1_data_wrangling.ipynb` → `2_berechnung.ipynb`

**Danach**
- Blog-Plots: `d1_empfehlungen_zeit.png`, `d4_abstimmungen_zeit.png`, `d3_kongruenz_zeit.html`
- Kein Pflicht-Folgenotebook; parallel zu `3a_`, `3c_*`, `3d_`

## Setup


## Import


Bibliotheken und Plot-Hilfen laden (`visualisierungen` + Archiv-Funktionen).


In [ ]:
# Pakete laden – autoreload ist praktisch beim Entwickeln,
# damit Änderungen an visualisierungen.py sofort übernommen werden
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
from visualisierungen import *


## Daten laden

Datensatz mit Kongruenzwerten einlesen.


In [ ]:
# Aufbereiteter Datensatz laden – enthält bereits die berechneten Positionen
df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.head())

Zwischenstand: DataFrame kurz in der Ausgabe anzeigen.


In [ ]:
df

Kontrolle 19. Jahrhundert: Wie oft lag die BV-Empfehlung mit dem Volk?


In [ ]:
# Schnellcheck: Wie oft lag der Bund im 19. Jahrhundert richtig?
# Werte über 0 = Bund und Volk waren gleicher Meinung
kontrolle = df[df['jahr'] < 1900]

# Gewonnene Abstimmungen (BV-Empfehlung stimmt mit Volk überein)
anzahl_gewonnen = len(kontrolle[kontrolle['zustimmung_bv-pos'] > 0])

# Verlorene Abstimmungen (BV-Empfehlung weicht vom Volk ab)
anzahl_verloren = len(kontrolle[kontrolle['zustimmung_bv-pos'] < 0])
anteil_verloren = 1-(anzahl_gewonnen /(anzahl_gewonnen + anzahl_verloren))
print(f"Gewonnen: {anzahl_gewonnen}")
print(f"Verloren: {anzahl_verloren}")
print(f"Anteil Verloren: {anteil_verloren}")

Gleiche Kontrolle für die erste Hälfte des 20. Jahrhunderts.


In [ ]:
# Gleiche Analyse für die erste Hälfte des 20. Jahrhunderts
kontrolle = df[(df['jahr'] >= 1900) & (df['jahr'] < 1949)]

# Gewonnene Abstimmungen (BV-Empfehlung stimmt mit Volk überein)
anzahl_gewonnen = len(kontrolle[kontrolle['zustimmung_bv-pos'] > 0])

# Verlorene Abstimmungen (BV-Empfehlung weicht vom Volk ab)
anzahl_verloren = len(kontrolle[kontrolle['zustimmung_bv-pos'] < 0])
anteil_verloren = 1-(anzahl_gewonnen /(anzahl_gewonnen + anzahl_verloren))
print(f"Gewonnen: {anzahl_gewonnen}")
print(f"Verloren: {anzahl_verloren}")
print(f"Anteil: {anteil_verloren}")

## Vorbereitung

Spalten und Gruppen für Institutionen/Parteien festlegen.


In [ ]:
# Spalten der Parteipositionen – wird unten oft wiederverwendet
partei_cols = ['zustimmung_p-fdp', 'zustimmung_p-sps', 'zustimmung_p-svp', 'zustimmung_p-mitte', 'zustimmung_p-gps']

# Übersicht: welche Akteure schauen wir an, und welche Spalten gehören dazu
institutionen = {
    'Bundesrat': ['br-pos_label'],
    'Bundesversammlung': ['bv-pos_label'],
    'Parteien': ['p-fdp_label', 'p-sps_label', 'p-svp_label', 'p-mitte_label', 'p-gps_label']
}

## Überblick: Abstimmungen und Empfehlungen

Wie viele Volksabstimmungen gab es – und wie aktiv haben Akteure Empfehlungen abgegeben?


### Balkendiagramm: Anzahl Abstimmungen über Zeit

In [ ]:
# Abstimmungen pro Jahrzehnt und Rechtsform zählen
anzahl_vote = df.groupby(['jahrzehnt', 'rechtsform_name']).size().reset_index(name='anzahl')
print(anzahl_vote)

# In breites Format bringen damit das gestapelte Balkendiagramm funktioniert
pivot = anzahl_vote.pivot(index='jahrzehnt', columns='rechtsform_name', values='anzahl').fillna(0)
# Jahrzehnte ohne Abstimmungen mit 0 auffüllen
pivot = pivot.reindex(range(1840, 2030, 10), fill_value=0)
fig = gestapeltes_balkendiagramm(
    pivot,
    xlabel="", ylabel="Anzahl", figsize=(12, 6))

plt.savefig("../Blog/blog_plots/d4_abstimmungen_zeit.png", dpi=300, bbox_inches='tight')

### Balkendiagramm (Anteil): Wer gab Abstimmungsempfehlungen wann?

In [ ]:
# Empfehlungen die wir als "klare Position" zählen
gezaehlt = ["Befürwortend", "Ablehnend", "Leer einlegen", "Vorzug Gegenentwurf", "Vorzug Volksinitiative"]

# Labels wo das Organ überhaupt existierte – wichtig als Nenner der Anteilsberechnung
# "Existiert nicht" (9999) und NaN bleiben in beiden Listen aussen vor
existiert_labels = [
    "Befürwortend",
    "Ablehnend",
    "Keine Empfehlung",
    "Leer einlegen",
    "Stimmfreigabe",
    "Vorzug Gegenentwurf",
    "Vorzug Volksinitiative",
    "Neutral",
]

frames = []
for gruppe, cols in institutionen.items():
    if cols in ("br-pos_label", "bv-pos_label"):
        # BR und BV gab es schon immer – einfach Anteil non-NaN als Aktivitätsgrad nehmen
        anteil = df.groupby('jahrzehnt')[cols].apply(lambda x: x.notna().median())
        temp = anteil.reset_index(name='anteil')
    else:
        # Bei Parteien aufpassen: manche gab es in früheren Jahrzehnten noch nicht
        # → nur Zeilen zählen wo das Label in der existiert_labels-Liste steht
        empfehlungen = df.groupby('jahrzehnt')[cols].apply(
            lambda x: x.isin(gezaehlt).sum().sum()
        )
        existiert = df.groupby('jahrzehnt')[cols].apply(
            lambda x: x.isin(existiert_labels).sum().sum()
        )
        # Anteil = konkrete Empfehlungen / alle Fälle wo Partei aktiv war
        temp = (empfehlungen / existiert).reset_index(name='anteil')
    temp['gruppe'] = gruppe
    frames.append(temp)

numbers_vote = pd.concat(frames, ignore_index=True)

# Kurzer Kontrollwert: wie hoch ist der Mean-Anteil bei Parteien ab 1900?
print(numbers_vote[(numbers_vote['gruppe'] == "Parteien") & (numbers_vote['jahrzehnt'].astype(int) > 1890)]['anteil'].median())

Empfehlungsanteile plotten und als `d1_empfehlungen_zeit.png` exportieren.


In [ ]:
# Jahrzehnte ohne Abstimmungen mit 0 auffüllen
numbers_vote = (
    numbers_vote.pivot(index='jahrzehnt', columns='gruppe', values='anteil')
    .reindex(range(1840, 2030, 10), fill_value=0)
    .reset_index()
    .melt(id_vars='jahrzehnt', var_name='gruppe', value_name='anteil')
)
# Diagramm ausgeben und für den Blog speichern
balkendiagramm(numbers_vote, x='jahrzehnt', y='anteil', hue='gruppe', palette=PALETTE_KATEGORIAL,
               titel="Anteil abgegebener Empfehlungen pro Jahrzehnt",
               xlabel="Jahrzehnt", ylabel="Anteil",
               figsize=(12, 6), rotation=45)
plt.savefig("../Blog/blog_plots/d1_empfehlungen_zeit.png", dpi=300, bbox_inches='tight')

Zwischenstand: DataFrame kurz in der Ausgabe anzeigen.


In [ ]:
df

## Kongruenz im Zeitverlauf

Entwicklung der Übereinstimmung mit Volk pro Institution und Partei.


## Postion Insitution vs. Position Bevölkerung

### BR

#### Liniendiagramm: BR vs. Bevölkerung

Linienplot: Übereinstimmung mit dem Bundesrat über Legislaturperioden.


In [ ]:
# Schnelle Übersicht: wie entwickelt sich die Übereinstimmung mit dem Bundesrat über die Zeit?
# errorbar=True zeigt die Streuung – gibt Hinweise auf Phasen mit vielen Ausreissern
liniendiagramm(df, x="legisjahr", y="zustimmung_br-pos", xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, errorbar=True, hline=0)

#### Boxplots+Lineplot: BR vs. Bevölkerung

Boxplots pro Jahr plus Mittelwert-Linie pro Legislatur (Bundesrat).


In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Boxplots zeigen die Verteilung aller Abstimmungen pro Abstimmungsjahr
sns.boxplot(data=df, x="jahr", y="zustimmung_br-pos", ax=ax, color=HAUPTFARBE)

# Zusätzlich Mean pro Legislatur drüber legen – das glättet den Verlauf
# und macht langfristige Trends sichtbarer
legis_mean = df.groupby("legisjahr")["zustimmung_br-pos"].mean().reset_index()

# Problem: der Boxplot hat eine kategorische x-Achse (ein Tick pro Jahr)
# → wir müssen die Legislaturperiode auf den richtigen Tick-Index mappen
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    # Mittleres Jahr der Legislaturperiode bestimmen für die x-Position
    matching = df[df["legisjahr"] == row["legisjahr"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_br-pos"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit Bundesrat pro Legislatur")

ax.axhline(0, color="grey", linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2026")
ax.set_xlim(left="1848", right="2026")
ax.set_ylim(bottom=-0.5, top=0.5)

ax.legend()
plt.tight_layout()
plt.show()

### BV

#### Liniendiagramm: BV vs. Bevölkerung

Linienplot: BV-Kongruenz in 5-Jahres-Blöcken.


In [ ]:
# Gleiche Übersicht für die Bundesversammlung, hier in 5-Jahres-Schritten aggregiert
liniendiagramm(df, x="5_jahre", y="zustimmung_bv-pos", xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, errorbar=True,  hline=0)

#### Boxplots+Lineplot: BV vs. Bevölkerung

Boxplots + Trendlinie für die Bundesversammlung.


In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Boxplots zeigen die Verteilung aller Abstimmungen pro Jahr
sns.boxplot(data=df, x="jahr", y="zustimmung_bv-pos", ax=ax, color=HAUPTFARBE)

# Mean pro 5-Jahres-Periode als Linie – glättet kurzfristige Schwankungen
legis = df.groupby("5_jahre")["zustimmung_bv-pos"].mean().reset_index()

# Gleiche Mapping-Logik wie beim BR: kategorische x-Achse auf Tick-Index mappen
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    matching = df[df["5_jahre"] == row["5_jahre"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_bv-pos"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit Bundesversammlung pro Legislatur")

ax.axhline(0, color=AKZENTFARBE, linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2026")
ax.set_xlim(left="1848", right="2026")
ax.set_ylim(bottom=-0.5, top=0.5)

ax.legend()
plt.tight_layout()
plt.show()

### Parteien

Alle Hauptparteien in einem Liniendiagramm (Jahrzehnt).


In [ ]:
# Parteispalten in Langformat bringen damit alle Parteien auf einer Achse geplottet werden können
partei_cols = ["zustimmung_p-svp", "zustimmung_p-fdp", "zustimmung_p-mitte", "zustimmung_p-sps", "zustimmung_p-gps"]

df_long = df[["jahrzehnt"] + partei_cols].melt(
    id_vars="jahrzehnt", var_name="partei", value_name="zustimmung")
# Spaltennamen auf lesbare Parteinamen kürzen
df_long['partei'] = df_long['partei'].str.replace('zustimmung_p-', '').str.upper()

liniendiagramm(df_long, x="jahrzehnt", y="zustimmung", hue="partei",
               xlabel="Jahrzehnt", ylabel="Übereinstimmung", rotation=90, hline=0)

#### Boxplot+Lineplot: Individuelle Parteien

Beispiel einzelne Partei (SP): Boxplots mit 5-Jahres-Mittel.


In [ ]:
fig, ax = plt.subplots(figsize=(20, 5))

# Beispiel für eine einzelne Partei – hier SP
sns.boxplot(data=df, x="jahr", y="zustimmung_p-sps", ax=ax, color=HAUPTFARBE)

# Mean pro 5-Jahres-Periode als Trendlinie
legis_mean = df.groupby("5_jahre")["zustimmung_p-sps"].mean().reset_index()

# Gleiche Mapping-Logik wie bei BR/BV
jahr_labels = [t.get_text() for t in ax.get_xticklabels()]
legis_x = []
legis_y = []
for _, row in legis_mean.iterrows():
    matching = df[df["5_jahre"] == row["5_jahre"]]["jahr"]
    mid_jahr = str(int(matching.mean()))
    if mid_jahr in jahr_labels:
        legis_x.append(jahr_labels.index(mid_jahr))
        legis_y.append(row["zustimmung_p-sps"])

ax.plot(legis_x, legis_y, color=AKZENTFARBE, linewidth=2, marker="o", label="Durchschnittliche Zustimmung mit SP pro Legislatur")

ax.axhline(0, color=AKZENTFARBE, linestyle="--")
ax.set_xlabel("Jahr")
ax.set_ylabel("Übereinstimmung")
ax.tick_params(axis="x", rotation=90)

# x-Achse ab 1848 begrenzen (Index in den kategorialen Labels finden)
ax.set_xlim(left="1848", right="2026")
ax.set_xlim(left="1848", right="2026")
ax.set_ylim(bottom=-0.5, top=0.5)

ax.legend()
plt.tight_layout()
plt.show()

Kontrolle: wie viele Abstimmungen pro 5-Jahres-Block.


In [ ]:
print(df['5_jahre'].value_counts().sort_index())

## Vergleich BR, BV und Parteien


BR, BV und Parteien-Mittel in einem Vergleichs-Liniendiagramm.


In [ ]:
bund = df[['legisjahr', 'zustimmung_br-pos', 'zustimmung_bv-pos'] + partei_cols].copy()
# Parteien zu einem einzigen Mean-Wert zusammenfassen (über alle Parteispalten pro Zeile)
bund['zustimmung_parteien'] = bund[partei_cols].mean(axis=1)

# In Langformat bringen für den Vergleichsplot
bund = bund[['legisjahr', 'zustimmung_br-pos', 'zustimmung_bv-pos', 'zustimmung_parteien']].melt(
    id_vars='legisjahr', var_name='institution', value_name='zustimmung')

liniendiagramm(bund, x="legisjahr", y="zustimmung", hue="institution",
               xlabel="Legislatur", ylabel="Übereinstimmung", rotation=45, hline=0)

Dieser Plot ist spannend. Ab den 1970er Jahren ist die Zustimmung der Bevölkerung mit Bund/Parlament sehr viel höher als mit den Parteien. Ausserdem BV und BR sehr deckungsgleich ab 1980?
Gab es einen instiutionellen Wandel der diese Entwicklung erklärt? Einfluss Frauenstimmrecht? Später Aufschwung SVP? Uniforme Position
Einführung Zauberformel 1959 --> Erhöhte Einigkeit zwischen BV und BR

## Nach Rechtsform

Unterscheidet sich die Kongruenz bei Referendum vs. Initiative?


## BR

Scatter: BR-Kongruenz nach Rechtsform und Jahrzehnt.


In [ ]:
# Gibt es Unterschiede je nach Rechtsform?
# Obligatorische Referenden betreffen oft Verfassungsänderungen – spannend ob Bund da öfter gewinnt
scatterplot(df, x="jahrzehnt", y="zustimmung_br-pos", size=None, titel="Zustimmung mit Bundesrat nach Rechtsform", xlabel="", ylabel="", legendentitel='Rechtsform',
               sizes=(20,800), alpha=0.8, hue='rechtsform_name', figsize=(10,5), rotation=90)

Linienplot: BR-Kongruenz nach Rechtsform über die Zeit.


In [ ]:
# Als Liniendiagramm – übersichtlicher für langfristige Trends nach Rechtsform
liniendiagramm(df, x="jahrzehnt", y="zustimmung_br-pos",
               xlabel="Jahrzehnt", ylabel="Übereinstimmung mit Bundesversammlung", rotation=90, hue="rechtsform_name", errorbar=False, hline=0)

## BV

Linienplot: BV-Kongruenz nach Rechtsform.


In [ ]:
# Gleiche Aufschlüsselung für die Bundesversammlung
liniendiagramm(df, x="jahrzehnt", y="zustimmung_bv-pos",
               xlabel="Jahr", ylabel="Übereinstimmung", rotation=90, hue="rechtsform_name", errorbar=None,  hline=0)

#### Legislaturjahre

In [ ]:
# Anteile der Rechtsformen pro Jahrzehnt berechnen
rechtsform = df.groupby('jahrzehnt')['rechtsform_name'].value_counts(normalize=True).reset_index(name='anzahl')

# Das Jahrzehnt 1850 fehlt im Datensatz – leere Zeile einfügen damit die x-Achse lückenlos bleibt
neue_zeilen = pd.DataFrame({
    'jahrzehnt': [1850],
    'rechtsform_name': ['Keine Abstimmungen'],
    'anzahl': [0]
})

rechtsform = pd.concat([rechtsform, neue_zeilen], ignore_index=True).sort_values('jahrzehnt').reset_index(drop=True)

Gestapeltes Balkendiagramm: Rechtsformen im Zeitverlauf.


In [ ]:
# Gestapeltes Balkendiagramm der Rechtsformen über die Zeit
balkendiagramm(data=rechtsform, x='jahrzehnt', y='anzahl', hue='rechtsform_name', palette=PALETTE_KATEGORIAL)

Zwischenstand: DataFrame kurz in der Ausgabe anzeigen.


In [ ]:
df

## Interaktive Grafik (Blog)

Zeitreihe mit Zeitraum- und Akteurswahl für die Website.


Plotly im Browser öffnen (praktischer für interaktive Grafiken).


In [ ]:
# Plotly im Browser öffnen statt im Notebook – besser für interaktive Grafiken
import plotly.io as pio
pio.renderers.default = "browser"
print(pio.renderers.default)

Interaktive Zeitreihe exportieren als `d3_kongruenz_zeit.html` für den Blog.


In [ ]:
# Spalten für alle Akteure die wir in der interaktiven Grafik zeigen wollen
akteur_cols = [
    'zustimmung_br-pos', 'zustimmung_bv-pos',
    'zustimmung_p-gps',    # Grüne
    'zustimmung_p-sps',    # SP
    'zustimmung_p-mitte',  # Mitte
    'zustimmung_p-fdp',    # FDP
    'zustimmung_p-svp',    # SVP
]

# Lesbare Namen für die Legende
label_map = {
    'zustimmung_br-pos':  'Bundesrat',
    'zustimmung_bv-pos':  'Bundesversammlung',
    'zustimmung_p-gps':   'Grüne',
    'zustimmung_p-sps':   'SP',
    'zustimmung_p-mitte': 'Mitte',
    'zustimmung_p-fdp':   'FDP',
    'zustimmung_p-svp':   'SVP',
}

# Farben angelehnt an die offiziellen Parteifarben (ungefähr)
farben_map = {
    'Bundesrat':         '#4477AA',
    'Bundesversammlung': '#CC6677',
    'Grüne':  '#999933',
    'SP':     '#882255',
    'Mitte':  '#DDCC77',
    'FDP':    '#332288',
    'SVP':    '#117733',
}

# Interaktive Grafik erstellen – Nutzer kann Zeitraum und Akteure selbst wählen
fig = liniendiagramm_interaktiv_zeitwahl(
    df,
    wert_cols=akteur_cols,
    label_map=label_map,
    farben_map=farben_map,
    titel='Übereinstimmung pro Akteur',
    xlabel='Jahr',
    ylabel='Übereinstimmung',
    yrange=(-0.5, 0.5))

fig.show()
# Als HTML speichern damit es im Blog eingebettet werden kann
fig.write_html(
    "../Blog/blog_plots/d3_kongruenz_zeit.html",
    include_plotlyjs='inline',
    full_html=True,
)

Zwischenstand: DataFrame kurz in der Ausgabe anzeigen.


In [ ]:
df